In [ ]:
## In this example, we will study output data frame from pandora.py configuration
#### 1. Opening each data frame and check structure
#### 2. Collect POT and scale factor to the target POT
#### 3. Merge evtdf and mcnudf for further study
#### 4. Draw some plots for each slice and for each pfp

import os
import sys
print('Importing cafpyana utils...')
cafpyana_root = "/home/lpelegri/cafpyana"
# Add CAFpyana to the Python search path -- this will allow you to import cafpyana modules
sys.path.insert(0, cafpyana_root)

import lmfit
import numpy as np
import math
import uproot as uproot
import pickle
import pandas as pd

print("Loading")
import matplotlib.pyplot as plt
import matplotlib.colors
from matplotlib.colors import LinearSegmentedColormap
from matplotlib import ticker
from matplotlib.ticker import (AutoMinorLocator, MultipleLocator)
from matplotlib import gridspec

from analysis_village.unfolding.wienersvd import *
from analysis_village.unfolding.unfolding_inputs import *

# Add the head direcoty to sys.path
workspace_root = os.getcwd()  
sys.path.insert(0, workspace_root + "/../../")


from analysis_village.cc1pi.var_configs import *

from analysis_village.cc1pi.HelperFunctions import HelperFunctions
from analysis_village.cc1pi.Constants import CTE as CTE
from analysis_village.cc1pi.CutMasks.CutMasks import *
from analysis_village.cc1pi.CutMasks.MaskUtils import *
from analysis_village.cc1pi.DataFrameUtils import DFCleaning
from analysis_village.cc1pi.DataFrameUtils.DFLoading import *
from analysis_village.cc1pi.GraphUtils.GraphsUtils import *
from analysis_village.cc1pi.GraphUtils.Utils import *

# import this repo's classes
import pyanalib.pandas_helpers as ph
import pyanalib.split_df_helpers as splh
import pyanalib.stat_helpers as sh

from pyanalib.split_df_helpers import *
from analysis_village.cc1pi.systematics.final_variable_configs import VariableConfig
from analysis_village.cc1pi.systematics.utils import *
from analysis_village.cc1pi.systematics.constants import *
from pyanalib.covariance import *
from analysis_village.cc1pi.DataFrameUtils.DFLoading import *
from cols_to_keep import *

from analysis_village.cc1pi.TLExtensionMethod.GaussianFactorFittingUtils import *
from analysis_village.cc1pi.TLExtensionMethod.ConvolutionTestUtils import *
from analysis_village.cc1pi.TLExtensionMethod.PlottingUtils import *

np.seterr(divide='ignore', invalid='ignore', over='ignore')

In [ ]:
import os, glob

base_path = "."  # match your existing base_path

# Remove every ACLiC-generated artifact for both classes
patterns = [
    "PhysdEdx_cpp*", "Hypfit_cpp*",
    "PhysdEdx_ACLiC_dict*", "Hypfit_ACLiC_dict*",
    "*.pcm", "*.d",
]
for pat in patterns:
    for f in glob.glob(os.path.join(base_path, pat)):
        print("removing", f)
        os.remove(f)

In [ ]:
# ======================================================================
# Configuration
# ======================================================================
import os
from pathlib import Path

PARTICLE = "pion"                                   # "pion" or "muon"
PDG = {"pion": 211, "muon": 13}[PARTICLE]

CAF_DIR = Path("/exp/sbnd/data/users/lpelegri/TLECafpyanaData_gen1")
INPUT_FILES = {
    "muon": {"MC": "TLE_1e20_muon_update_calo.df",
             "Data": "TLE_5e18_muon_data_update_calo.df"},
    "pion": {"MC": "TLE_1e20_pion_stopping_update_calo.df",
             "Data": "TLE_1e20_pion_data_update_calo.df"},
}
LOAD_NFILES = {"MC": 30, "Data": 100}               # last argument of load_df
KEYS2LOAD = ["pfp", "hdr", "histpotdf", "hit0", "hit1", "hit2"]

PHYS_DIR = Path("/home/lpelegri/cafpyana/analysis_village/cc1pi/TLExtensionMethod")
TLE_DIR = Path("/exp/sbnd/data/users/lpelegri/TLE_processed_files")   # existing MC variation outputs

# ---- dE/dx slice fits and parametrizations ----
SLICE_RR_MAX = {"muon": 250.0, "pion": 80.0}       # upper residual range of the slice fits [cm]
SLICE_SIGMA_ERR_RANGE = (1e-4, 0.2)                # slices kept: gsigma_err inside this range
SLICE_PITCH = 0.32                                 # [cm], same pitch as the convolution maps
PION_SHIFT_FROM_MUONS = True                       # pions: beyond PION_RR_JOIN use muon slices at equal βγ
PION_RR_JOIN = 80.0                                # [cm]
PION_OVERLAP = (40.0, 80.0)                        # pion residual range used to match the muon offset [cm]
PARAM_TPC = -1                                     # TPC entry loaded into the per-plane C++ maps

# ---- Track processing ----
CHECK_PUR_COMP = False
MAKE_ENERGY_CUT = False
USE_BEST_PLANE = True
SELECTED_PLANE = 2
SUBTRACT_HITS = True
N_SKIP = 10 if PDG == 211 else 25
N_CORES = max(1, int(os.cpu_count() * 0.8))
MAX_EXTENSION = 300.0   # [cm], only used if Hypfit does not expose max_additional_res_length_pion
MAX_RR_CAP = 800        # [cm], upper limit on the length of the convolution maps

# Column names in the pfp DataFrame
COLS = {
    "true_P": "true_P",
    "range_P": "range_P",
    "length": "length",
    "purity": "purity",
    "completeness": "completeness",
    "PDG": "PDG",
    "end_process": "end_process_string",
    "best_plane": "best_plane",
}

PRESETS = [
    {"preset_key": "range", "method": "range",
     "cleaning_method": "skip_only", "normalize": False, "include_small_likelihood": False},
    {"preset_key": "original", "method": "likelihood",
     "cleaning_method": "skip_only_truncate_harsh", "normalize": False, "include_small_likelihood": False},
    {"preset_key": "normalize_clean_dEdx_include_small", "method": "likelihood",
     "cleaning_method": "all", "normalize": True, "include_small_likelihood": True},
    {"preset_key": "convolution_normalize_clean_dEdx_include_small_og", "method": "convolution_og",
     "cleaning_method": "all", "normalize": True, "include_small_likelihood": True},
    {"preset_key": "convolution_normalize_clean_dEdx_include_small_shifted", "method": "convolution_shifted",
     "cleaning_method": "all", "normalize": True, "include_small_likelihood": True},
]
SHIFTED_KEY = "convolution_normalize_clean_dEdx_include_small_shifted"
OG_KEY = "convolution_normalize_clean_dEdx_include_small_og"

# ---- Tests ----
USE_RANGE_MOMENTUM = True          # reference momentum: range_P (True) or true_P (False)
HITS_TO_PLOT = "percent"           # "percent", "used" or "removed"
HIT_BINS_PERCENT = [0, 0.25, 0.5, 0.65, 0.8, 0.9]
HIT_BINS_COUNTS = [0, 50, 100, 200, 300, 500]

# MC variation outputs in TLE_DIR (read only, as they are)
VARIATION_FILES = {
    "muon": ["13_subtracted_TLE_1e20_muon_alpha_m",
             "13_subtracted_TLE_1e20_muon_alpha_p",
             "13_subtracted_TLE_1e20_muon_ccal_m",
             "13_subtracted_TLE_1e20_muon_ccal_p",
             "13_subtracted_TLE_wiremod_YZ_muon"],
    "pion": ["211_subtracted_TLE_1e20_pion_stopping_alpha_m",
             "211_subtracted_TLE_1e20_pion_stopping_alpha_p",
             "211_subtracted_TLE_1e20_pion_stopping_ccal_m",
             "211_subtracted_TLE_1e20_pion_stopping_ccal_p",
             "211_subtracted_TLE_wiremod_YZ_pion_stopping_update_calo"],
}[PARTICLE]

FRAC_MAX = 0.25     # tail test: upper edge of the lowest track-fraction bin
RES_MIN = 0.3       # tail test: residual above which a track counts as tail

In [ ]:
# ======================================================================
# Physics classes (PhysdEdx, Hypfit)
# ======================================================================
def load_physics_classes(base_dir=PHYS_DIR):
    """Compiles (if needed) and loads PhysdEdx and Hypfit; returns a Hypfit instance.
    Compiler messages go to the terminal running the Jupyter kernel, not to the cell."""
    for src in ("PhysdEdx.cpp", "Hypfit.cpp"):
        if ROOT.gSystem.CompileMacro(str(base_dir / src), "k") != 1:
            raise RuntimeError(f"Compilation of {src} failed; see the terminal running the kernel")
    return ROOT.Hypfit()


h_fit = load_physics_classes()
PDG_MASS = {13: 105.6583755, 211: 139.57039, 2212: 938.27208816}

In [ ]:
# ======================================================================
# Input samples (cafpyana DataFrames)
# ======================================================================
_LOADED = {}


def get_sample(particle, sample):
    """pfp DataFrame and [hit0, hit1, hit2] DataFrames of one input file, loaded once."""
    key = (particle, sample)
    if key not in _LOADED:
        path = CAF_DIR / INPUT_FILES[particle][sample]
        print(f"Loading {path}")
        dfs = load_df(str(path), KEYS2LOAD, LOAD_NFILES[sample])
        _LOADED[key] = {"pfp": dfs["pfp"], "hits": [dfs["hit0"], dfs["hit1"], dfs["hit2"]]}
    return _LOADED[key]


def drop_sample(particle, sample):
    """Frees the memory of a loaded sample."""
    _LOADED.pop((particle, sample), None)
    gc.collect()

In [ ]:
import gc
import multiprocessing as mp
import pickle
import queue as _queue
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import ROOT
from scipy.optimize import minimize
from scipy.signal import fftconvolve
from tqdm.auto import tqdm

# Import external helper functions from analysis_village
if "load_df" not in globals():
    from analysis_village.cc1pi.DataFrameUtils.DFLoading import load_df

import analysis_village.cc1pi.TLExtensionMethod.GaussianFactorFittingUtils as _lf
for _name in ("load_physics_classes", "make_theoretical_mpv_func", "make_rr_edges",
              "fit_param_curve", "power_law", "PARAM_SPECS"):
    if _name not in globals():
        globals()[_name] = getattr(_lf, _name)

CAF_DIR = Path("/exp/sbnd/data/users/lpelegri/TLECafpyanaDataRecalo")
INPUT_FILES = {
    "pion": {"MC": "TLE_1e20_pion_stopping.df",
             "Data": "TLE_1e20_pion_data.df"},
    "muon": {"MC": "TLE_1e20_muon.df",
             "Data": "TLE_5e18_muon_data.df"},
}
LOAD_NFILES = {"MC": 30, "Data": 100}
HIT_KEYS = ["hit0", "hit1", "hit2"]
NEEDED_COLS = ["rr", "dedx", "pitch", "tpc"]

_HITS_CACHE = {}

def load_hits(particle, sample):
    """[hit0, hit1, hit2] of one sample, reduced to NEEDED_COLS. Only the most recently
    loaded sample is kept, so at most one sample is in memory at a time."""
    key = (particle, sample)
    if key not in _HITS_CACHE:
        _HITS_CACHE.clear()
        gc.collect()
        path = CAF_DIR / INPUT_FILES[particle][sample]
        print(f"Loading {path}")
        dfs = load_df(str(path), HIT_KEYS, LOAD_NFILES[sample])
        hits = []
        for k in HIT_KEYS:
            df = dfs[k]
            cols = [c for c in NEEDED_COLS if c in df.columns]
            hits.append(df[cols].reset_index(drop=True))
            print(f"  {k}: {len(hits[-1]):,} hits, {hits[-1].memory_usage(deep=True).sum() / 1e9:.2f} GB")
        del dfs
        gc.collect()
        _HITS_CACHE[key] = hits
    return _HITS_CACHE[key]

def _build_landau_table():
    lam = np.concatenate([np.arange(-8.0, 20.0, 0.005), np.arange(20.0, 1000.0, 0.05)])
    pdf = np.array([ROOT.TMath.Landau(float(l)) for l in lam])
    pdf /= np.sum(0.5 * (pdf[1:] + pdf[:-1]) * np.diff(lam))
    return lam, pdf

_LAM, _LAN = _build_landau_table()
_LAMBDA_MODE = float(_LAM[np.argmax(_LAN)])

def _make_grid(lo, hi, bin_width, sub=4):
    nbins = int(round((hi - lo) / bin_width))
    h = (hi - lo) / (nbins * sub)
    t = lo + (np.arange(nbins * sub) + 0.5) * h
    centers = lo + (np.arange(nbins) + 0.5) * (hi - lo) / nbins
    return nbins, sub, h, t, centers

def _bin_probs(loc, xi, sigma, grid):
    nbins, sub, h, t, _ = grid
    land = np.interp((t - loc) / xi, _LAM, _LAN, left=0.0, right=0.0) / xi
    k = max(int(np.ceil(5.0 * sigma / h)), 1)
    ker = np.exp(-0.5 * (np.arange(-k, k + 1) * h / sigma) ** 2)
    ker /= ker.sum()
    conv = fftconvolve(land, ker, mode="same")
    return conv.reshape(nbins, sub).sum(axis=1) * h

def _nll(loc, xi, sigma, counts, grid):
    p = _bin_probs(loc, xi, sigma, grid)
    s = p.sum()
    if not np.isfinite(s) or s <= 1e-12:
        return 1e30
    mu = np.maximum(counts.sum() * p / s, 1e-12)
    return float(np.sum(mu - counts * np.log(mu)))

def _covariance(f, th, steps):
    n = len(th)
    f0 = f(th)
    H = np.zeros((n, n))
    for i in range(n):
        ei = np.zeros(n)
        ei[i] = steps[i]
        H[i, i] = (f(th + ei) - 2 * f0 + f(th - ei)) / steps[i] ** 2
        for j in range(i + 1, n):
            ej = np.zeros(n)
            ej[j] = steps[j]
            H[i, j] = H[j, i] = (f(th + ei + ej) - f(th + ei - ej)
                                 - f(th - ei + ej) + f(th - ei - ej)) / (4 * steps[i] * steps[j])
    try:
        cov = np.linalg.inv(H)
    except np.linalg.LinAlgError:
        return None
    return cov if np.all(np.diag(cov) > 0) else None

def fit_langau_slice(counts, grid, sigma=None, mpv_convention="location"):
    nbins, sub, h, t, centers = grid
    free_sigma = sigma is None
    smooth = np.convolve(counts, np.ones(5) / 5.0, mode="same")
    x_peak = centers[int(np.argmax(smooth))]

    def nll(th):
        return _nll(th[0], th[1], th[2] if free_sigma else sigma, counts, grid)

    bounds = [(x_peak - 1.0, x_peak + 1.5), (0.01, 1.0)] + ([(0.02, 1.0)] if free_sigma else [])
    lo_b = np.array([b[0] for b in bounds])
    hi_b = np.array([b[1] for b in bounds])

    best = None
    for xi0, sg0 in ((0.07, 0.20), (0.15, 0.10)):
        x0 = [x_peak - _LAMBDA_MODE * xi0, xi0] + ([sg0] if free_sigma else [])
        r = minimize(nll, np.clip(x0, lo_b, hi_b), method="Nelder-Mead", bounds=bounds,
                     options=dict(xatol=1e-5, fatol=1e-6, maxiter=800))
        if best is None or r.fun < best.fun:
            best = r

    th = best.x
    loc, xi = th[0], th[1]
    sg = th[2] if free_sigma else float(sigma)
    at_bound = bool(np.any(th - lo_b < 1e-3 * (hi_b - lo_b)) or np.any(hi_b - th < 1e-3 * (hi_b - lo_b)))

    steps = [0.004, min(max(0.003, 0.05 * xi), 0.5 * xi)]
    if free_sigma:
        steps.append(min(max(0.005, 0.05 * sg), 0.5 * sg))
    cov = _covariance(nll, th, steps)

    p = _bin_probs(loc, xi, sg, grid)
    s = p.sum()
    mu = np.maximum(counts.sum() * p / s, 1e-12)
    m = counts > 0
    deviance = 2.0 * np.sum(counts[m] * np.log(counts[m] / mu[m]))

    c = _LAMBDA_MODE
    if mpv_convention == "peak":
        mpv = loc + c * xi
        mpv_err = (np.sqrt(max(cov[0, 0] + c * c * cov[1, 1] + 2 * c * cov[0, 1], 0.0))
                   if cov is not None else np.nan)
    else:
        mpv = loc
        mpv_err = np.sqrt(cov[0, 0]) if cov is not None else np.nan

    return dict(
        ok=bool(best.success and not at_bound and cov is not None),
        mpv=mpv, mpv_err=mpv_err,
        width=xi, width_err=np.sqrt(cov[1, 1]) if cov is not None else np.nan,
        gsigma=sg,
        gsigma_err=(np.sqrt(cov[2, 2]) if cov is not None else np.nan) if free_sigma else 0.0,
        area=counts.sum() / s,
        chi2=deviance, ndf=nbins - (len(th) + 1),
    )

def check_one_slice(df, rr_lo, rr_hi, old_row, dedx_col="dedx", fit_range=(0.5, 20.0),
                    bin_width=0.025, mpv_convention="location"):
    grid = _make_grid(fit_range[0], fit_range[1], bin_width)
    v = df.loc[(df["rr"] >= rr_lo) & (df["rr"] < rr_hi), dedx_col].to_numpy(dtype=float)
    counts, _ = np.histogram(v, bins=grid[0], range=fit_range)
    free = fit_langau_slice(counts, grid, mpv_convention=mpv_convention)
    fixed = fit_langau_slice(counts, grid, sigma=float(old_row["gsigma"]), mpv_convention=mpv_convention)
    print(f"{'':16s}{'MPV':>16s}{'width':>16s}{'gsigma':>16s}")
    print(f"{'old fit':16s}{old_row['mpv_reco']:16.4f}{old_row['width']:16.4f}{old_row['gsigma']:16.4f}")
    for name, f in (("new, all free", free), ("new, sigma=old", fixed)):
        print(f"{name:16s}{f['mpv']:9.4f}±{f['mpv_err']:.4f}{f['width']:9.4f}±{f['width_err']:.4f}"
              f"{f['gsigma']:9.4f}±{f['gsigma_err']:.4f}   ok={f['ok']}")

_CTX2 = {}

def _report(key, rr_c):
    sink = _CTX2.get("progress")
    if sink is None:
        return
    if hasattr(sink, "put"):
        sink.put((key, rr_c))
    else:
        sink(key, rr_c)

def _slice_task(key):
    plane, tpc = key
    c = _CTX2
    df = c["hit_dfs"][plane]
    cols = [x for x in ("rr", c["dedx_col"], "pitch") if x in df.columns]
    sub = df[cols] if tpc == -1 else df.loc[df["tpc"] == tpc, cols]
    if c["max_pitch"] is not None:
        sub = sub[sub["pitch"] <= c["max_pitch"]]
    sub = sub.sort_values("rr")
    rr = sub["rr"].to_numpy(dtype=float)
    dedx = sub[c["dedx_col"]].to_numpy(dtype=float)
    pitch = sub["pitch"].to_numpy(dtype=float) if "pitch" in sub else None

    fixed = c["sigma_popt"] is not None
    popt = c["sigma_popt"].get(key) if fixed else None
    if fixed and popt is None:
        return key, pd.DataFrame()

    edges = np.asarray(c["edges"])
    idx = np.searchsorted(rr, edges)
    rows = []
    for i, (lo, hi) in enumerate(zip(edges[:-1], edges[1:])):
        a, b = idx[i], idx[i + 1]
        rr_c = 0.5 * (lo + hi)
        _report(key, rr_c)
        if b - a < c["min_entries"]:
            continue
        counts, _ = np.histogram(dedx[a:b], bins=c["grid"][0], range=c["fit_range"])
        if counts.sum() < c["min_entries"]:
            continue

        sigma = max(float(c["sigma_eval"](rr_c, *popt)), 0.02) if fixed else None
        f = fit_langau_slice(counts, c["grid"], sigma=sigma, mpv_convention=c["mpv_convention"])
        if not f["ok"]:
            continue
        if not fixed and not (np.isfinite(f["gsigma_err"]) and 1e-4 <= f["gsigma_err"] <= c["max_gsigma_err"]):
            continue
        if c["max_width_rel_err"] is not None and not (f["width_err"] <= c["max_width_rel_err"] * f["width"]):
            continue

        if c["theoretical_mpv_func"] is not None and pitch is not None:
            mpv_x, mpv_x_err = c["theoretical_mpv_func"](rr_c, float(np.nanmean(pitch[a:b]))), 0.0
        else:
            mpv_x, mpv_x_err = f["mpv"], f["mpv_err"]

        rows.append(dict(
            plane=plane, tpc=tpc, rr_center=rr_c, n_hits=int(b - a),
            mpv_x=mpv_x, mpv_x_err=mpv_x_err, mpv_reco=f["mpv"], mpv_reco_err=f["mpv_err"],
            width=f["width"], width_err=f["width_err"], area=f["area"], area_err=np.nan,
            gsigma=f["gsigma"], gsigma_err=f["gsigma_err"], chi2=f["chi2"], ndf=f["ndf"],
            status=0, gsigma_fixed=fixed,
        ))
    return key, pd.DataFrame(rows)

def _tpc_label(tpc):
    return "comb" if tpc == -1 else str(tpc)

def _run_pool(keys, n_workers, label, progress=True):
    n_slices = len(_CTX2["edges"]) - 1
    overall = tqdm(total=len(keys) * n_slices, desc=label, position=0, disable=not progress)
    bars = {k: tqdm(total=n_slices, desc=f"{label} | plane {k[0]} tpc {_tpc_label(k[1])}",
                    position=i + 1, leave=False, disable=not progress)
            for i, k in enumerate(keys)}

    def tick(k, rr_c):
        bars[k].update(1)
        bars[k].set_postfix(rr=f"{rr_c:g} cm")
        overall.update(1)

    def finish(k):
        b = bars[k]
        missing = (b.total or 0) - b.n
        if missing > 0:
            b.update(missing)
            overall.update(missing)
        b.close()

    out = []
    try:
        if n_workers <= 1:
            _CTX2["progress"] = tick
            for k in keys:
                out.append(_slice_task(k))
                finish(k)
        else:
            ctx = mp.get_context("fork")
            q = ctx.Queue()
            _CTX2["progress"] = q
            with ctx.Pool(min(n_workers, len(keys))) as pool:
                asyncs = {k: pool.apply_async(_slice_task, (k,)) for k in keys}
                done = set()
                while len(done) < len(keys):
                    try:
                        k, rr_c = q.get(timeout=0.2)
                        if k not in done:
                            tick(k, rr_c)
                    except _queue.Empty:
                        pass
                    for k, a in asyncs.items():
                        if k not in done and a.ready():
                            out.append(a.get())
                            done.add(k)
                            finish(k)
            q.close()
    finally:
        _CTX2.pop("progress", None)
        for b in bars.values():
            b.close()
        overall.close()
    return dict(out)

def analyze_two_pass(
    hit_dfs, particle="muon", dedx_col="dedx", rr_max_by_particle=None,
    theoretical_mpv_func=None, out_prefix="langau_rr", verbose=True,
    fit_model="power_law", fit_rr_lo=5.0, rr_min=2.0, rr_bin_width=1.0,
    fit_range=(0.5, 20.0), bin_width=0.025, min_entries=100,
    max_gsigma_err=0.2, max_pitch=None, max_width_rel_err=None,
    mpv_convention="location", sigma_eval=None, n_workers=4,
    progress=True, tag="", make_summary_plots=True,
):
    sigma_eval = sigma_eval or power_law
    if rr_max_by_particle is None:
        rr_max_by_particle = {"muon": 80.0, "pion": 60.0, "proton": 60.0}
    rr_max = rr_max_by_particle.get(particle, 80.0)
    grid = _make_grid(fit_range[0], fit_range[1], bin_width)
    keys = [(p, t) for t in (-1, 0, 1) for p in range(len(hit_dfs))]

    _CTX2.clear()
    _CTX2.update(
        hit_dfs=hit_dfs, dedx_col=dedx_col, max_pitch=max_pitch, fit_range=fit_range, grid=grid,
        min_entries=min_entries, max_gsigma_err=max_gsigma_err, max_width_rel_err=max_width_rel_err,
        mpv_convention=mpv_convention, sigma_eval=sigma_eval, theoretical_mpv_func=theoretical_mpv_func,
    )
    try:
        _CTX2.update(edges=make_rr_edges(rr_min, rr_max, rr_bin_width), sigma_popt=None)
        res1 = _run_pool(keys, n_workers, f"{tag} pass 1".strip(), progress)

        sigma_fit = {}
        for key, r in res1.items():
            rf = r[r["rr_center"] >= fit_rr_lo] if len(r) else r
            sigma_fit[key] = (fit_param_curve(rf, "gsigma", "gsigma_err", rr_lo=fit_rr_lo, model=fit_model)
                              if len(rf) >= 3 else (None, None))

        _CTX2.update(edges=make_rr_edges(fit_rr_lo, rr_max, rr_bin_width),
                     sigma_popt={k: v[0] for k, v in sigma_fit.items()})
        res2 = _run_pool(keys, n_workers, f"{tag} pass 2".strip(), progress)
    finally:
        _CTX2.clear()

    all_results, fit_params = {}, {}
    for key in sorted(keys, key=lambda k: (k[0], [0, 1, -1].index(k[1]))):
        r1 = res1[key].assign(gsigma_fixed=False) if len(res1[key]) else res1[key]
        r2 = res2.get(key, pd.DataFrame())
        if len(r2):
            parts = [r1[r1["rr_center"] < fit_rr_lo], r2] if len(r1) else [r2]
            res = pd.concat(parts, ignore_index=True)
        else:
            res = r1
        all_results[key] = res
        if verbose:
            print(f"plane {key[0]}, tpc {key[1]}: pass 1 {len(res1[key])} slices, "
                  f"pass 2 {len(r2)} slices, merged {len(res)}")

        res_fit = res[res["rr_center"] >= fit_rr_lo] if len(res) else res
        plane_fits = {}
        if len(res_fit) >= 3:
            for p_col, p_err, _, _ in PARAM_SPECS:
                if p_col == "area":
                    continue
                if p_col == "gsigma":
                    plane_fits[p_col] = sigma_fit[key]
                else:
                    plane_fits[p_col] = fit_param_curve(res_fit, p_col, p_err,
                                                        rr_lo=fit_rr_lo, model=fit_model)
        fit_params[key] = plane_fits

    non_empty = [r.assign(plane=p, tpc=t) for (p, t), r in all_results.items() if len(r)]
    combined = pd.concat(non_empty, ignore_index=True) if non_empty else pd.DataFrame()
    if len(combined):
        combined.to_hdf(f"{out_prefix}_slices.h5", key="fits", mode="w",
                        format="table", complib="blosc", complevel=5)
    return all_results, fit_params, combined

def run_langau_slices(hit_dfs_for, hfit, rr_max=None, n_workers=4, **analysis_kwargs):
    rr_max = rr_max or {"pion": 80.0, "muon": 250.0}
    langau, langau_fits = {}, {}
    for particle, pdg in (("pion", 211), ("muon", 13)):
        theoretical_mpv = make_theoretical_mpv_func(hfit, pdg=pdg)
        for sample in ("MC", "Data"):
            print(f"[Langau] {particle} {sample}, rr < {rr_max[particle]:g} cm")
            results, fit_params, _ = analyze_two_pass(
                hit_dfs_for(particle, sample),
                particle=particle,
                theoretical_mpv_func=theoretical_mpv,
                rr_max_by_particle={particle: rr_max[particle]},
                out_prefix=f"langau_{particle}_{sample}",
                verbose=False,
                n_workers=n_workers,
                tag=f"{particle} {sample}",
                **analysis_kwargs,
            )
            langau[(particle, sample)] = results
            langau_fits[(particle, sample)] = fit_params
    return langau, langau_fits

MASS = {13: 105.6583755, 211: 139.57039}

LANGAU_PARAMS = [
    ("mpv_reco", "mpv_reco_err", "Langau MPV [MeV/cm]"),
    ("gsigma", "gsigma_err", r"Gaussian smearing $\sigma_G$ [MeV/cm]"),
    ("width", "width_err", r"Landau width $\xi$ [MeV/cm]"),
]

POINT_STYLES = {
    ("pion", "MC"): ("#d62728", "o", True, "Pion MC"),
    ("muon", "MC"): ("#d62728", "s", False, r"Muon MC (equal $\beta\gamma$)"),
    ("pion", "Data"): ("#1f77b4", "o", True, "Pion data"),
    ("muon", "Data"): ("#1f77b4", "s", False, r"Muon data (equal $\beta\gamma$)"),
}

def rr_to_betagamma(hfit, pdg, rr):
    phys = hfit.map_PhysdEdx[pdg]
    return np.array([phys.KEtoMomentum(phys.KEFromRangeSpline(float(r))) / MASS[pdg] for r in rr])

def betagamma_to_rr(hfit, pdg, bg):
    phys = hfit.map_PhysdEdx[pdg]
    return np.array([phys.RangeFromKESpline(phys.MomentumtoKE(float(b) * MASS[pdg])) for b in bg])

def langau_x_values(hfit, particle, res, x_axis):
    rr = res["rr_center"].to_numpy(dtype=float)
    pdg = 211 if particle == "pion" else 13
    bg = rr_to_betagamma(hfit, pdg, rr)
    if x_axis == "betagamma":
        return bg
    return rr if particle == "pion" else betagamma_to_rr(hfit, 211, bg)

def plot_langau_pion_muon_points(langau, hfit, tpc=-1, x_axis="rr", x_limits=None,
                                 max_rel_err=None, y_limits=None):
    xs = {}
    for (particle, sample), results in langau.items():
        for plane in range(3):
            res = results.get((plane, tpc))
            if res is not None and len(res):
                xs[(particle, sample, plane)] = langau_x_values(hfit, particle, res, x_axis)

    for p_col, p_err, ylabel in LANGAU_PARAMS:
        fig, axes = plt.subplots(1, 3, figsize=(18, 6), layout="constrained")
        for plane, ax in enumerate(axes):
            y_lo, y_hi = np.inf, -np.inf
            pion_x_max = -np.inf
            for (particle, sample), (color, marker, filled, label) in POINT_STYLES.items():
                res = langau.get((particle, sample), {}).get((plane, tpc))
                if res is None or not len(res) or p_col not in res:
                    continue
                x = xs[(particle, sample, plane)]
                y = res[p_col].to_numpy(dtype=float)
                e = res[p_err].to_numpy(dtype=float)
                sel = np.isfinite(x) & np.isfinite(y) & np.isfinite(e)
                if p_col == "width":
                    sel &= y > 0
                if max_rel_err is not None:
                    sel &= e <= max_rel_err * np.abs(y)
                if x_limits is not None:
                    sel &= (x >= x_limits[0]) & (x <= x_limits[1])
                if not sel.any():
                    continue
                ax.errorbar(x[sel], y[sel], yerr=e[sel], fmt=marker, ms=3.5, color=color,
                            mfc=color if filled else "none", elinewidth=0.7, capsize=0,
                            alpha=0.85, label=label)
                y_lo = min(y_lo, np.nanmin(y[sel] - e[sel]))
                y_hi = max(y_hi, np.nanmax(y[sel] + e[sel]))
                if particle == "pion":
                    pion_x_max = max(pion_x_max, x[sel].max())

            if np.isfinite(pion_x_max):
                ax.axvline(pion_x_max, color="gray", ls="-.", lw=1, label="End of pion slices")
            if x_limits is not None:
                ax.set_xlim(x_limits)
                if np.isfinite(y_lo) and np.isfinite(y_hi):
                    pad = 0.08 * (y_hi - y_lo)
                    ax.set_ylim(y_lo - pad, y_hi + pad)
            if y_limits and p_col in y_limits:
                ax.set_ylim(y_limits[p_col])
            if x_axis == "betagamma":
                ax.set_xscale("log")
                ax.set_xlabel(r"$\beta\gamma$", fontsize=16)
            else:
                ax.set_xlabel("Pion residual range [cm]", fontsize=16)
            ax.set_title(f"Plane {plane}", fontsize=18)
            ax.tick_params(labelsize=13)
            ax.grid(True, ls=":", alpha=0.5)
            ax.set_box_aspect(1)
        axes[0].set_ylabel(ylabel, fontsize=16)
        axes[0].legend(fontsize=11, loc="upper right", framealpha=0.9)
        plt.show()

In [ ]:


def load_hits(particle, sample):
    """[hit0, hit1, hit2] of one sample. Only the most recently loaded sample is kept,
    so at most one sample is in memory at a time."""
    key = (particle, sample)
    if key not in _HITS_CACHE:
        _HITS_CACHE.clear()
        gc.collect()
        path = CAF_DIR / INPUT_FILES[particle][sample]
        print(f"Loading {path}")
        dfs = load_df(str(path), HIT_KEYS, LOAD_NFILES[sample])
        _HITS_CACHE[key] = [dfs["hit0"], dfs["hit1"], dfs["hit2"]]
    return _HITS_CACHE[key]

# cell 1, before importing numpy
import os
os.environ["OMP_NUM_THREADS"] = "1"; os.environ["MKL_NUM_THREADS"] = "1"

# cell 3: loads one sample at a time and fits it
hfit = load_physics_classes()
langau, langau_fits = run_langau_slices(load_hits, hfit, n_workers=6)

# cell 4: save, so you do not have to refit
import pickle
with open("langau_results.pkl", "wb") as f:
    pickle.dump(dict(langau=langau, langau_fits=langau_fits), f)

# cell 5: plots
plot_langau_pion_muon_points(langau, hfit)
plot_langau_pion_muon_points(langau, hfit, x_limits=(20, 340),
                             y_limits={"width": (0.0, 0.2), "gsigma": (0.0, 0.4)})
plot_langau_pion_muon_points(langau, hfit, x_axis="betagamma")

In [ ]:
# ======================================================================
# Muon Langau parameters moved onto the pion at equal betagamma by a constant offset
# Run after run_langau_slices (needs langau, hfit, langau_x_values).
# ======================================================================
OFFSET_PARAMS = [
    ("mpv_reco", "mpv_reco_err", "Langau MPV [MeV/cm]"),
    ("mpv_shift", "mpv_reco_err", r"Langau MPV $-$ theoretical MPV [MeV/cm]"),
    ("width", "width_err", r"Landau width $\xi$ [MeV/cm]"),
]

# MPV relative to the theoretical MPV of each slice (mpv_x), the analogue of the shift
for _results in langau.values():
    for _res in _results.values():
        if len(_res) and "mpv_x" in _res:
            _res["mpv_shift"] = _res["mpv_reco"] - _res["mpv_x"]


def _points(res, x, p_col, p_err, max_rel_err=None):
    """(x, y, error) of the slices with a finite value and a positive error, sorted in x.
    Slices where p_col was fixed in the fit (error 0) are dropped."""
    y = res[p_col].to_numpy(dtype=float)
    e = res[p_err].to_numpy(dtype=float)
    ok = np.isfinite(x) & np.isfinite(y) & np.isfinite(e) & (e > 0)
    if p_col == "width":
        ok &= y > 0
    if max_rel_err is not None:
        ok &= e <= max_rel_err * np.abs(y)
    order = np.argsort(x[ok])
    return x[ok][order], y[ok][order], e[ok][order]


def pion_muon_offset(hfit, pi_res, mu_res, p_col, p_err, window=None, max_rel_err=None):
    """Constant offset that moves the muon values of p_col onto the pion values.
    The muon slices are placed at the pion residual range of equal betagamma and interpolated
    at the pion slices inside the muon range (the overlap). The offset is the weighted mean
    pion - muon difference over the overlap, or over `window` (pion residual range [cm]) if
    given, with weights 1 / (pion error^2 + muon error^2). chi2 and ndf compare the pion with
    the offset muon over the whole overlap. None if there is no overlap."""
    pion = _points(pi_res, pi_res["rr_center"].to_numpy(dtype=float), p_col, p_err, max_rel_err)
    muon = _points(mu_res, langau_x_values(hfit, "muon", mu_res, "rr"), p_col, p_err, max_rel_err)
    (x_pi, y_pi, e_pi), (x_mu, y_mu, e_mu) = pion, muon
    if len(x_mu) < 2:
        return None
    ov = (x_pi >= x_mu[0]) & (x_pi <= x_mu[-1])
    x = x_pi[ov]
    diff = y_pi[ov] - np.interp(x, x_mu, y_mu)
    diff_err = np.sqrt(e_pi[ov] ** 2 + np.interp(x, x_mu, e_mu) ** 2)
    win = np.ones(len(x), dtype=bool) if window is None else (x >= window[0]) & (x <= window[1])
    if win.sum() < 2:
        return None
    w = 1.0 / diff_err[win] ** 2
    offset = float(np.sum(w * diff[win]) / np.sum(w))
    chi2 = float(np.sum(((diff - offset) / diff_err) ** 2))
    return dict(offset=offset, offset_err=float(1.0 / np.sqrt(np.sum(w))),
                chi2=chi2, ndf=len(x) - 1, n_window=int(win.sum()),
                x=x, diff=diff, diff_err=diff_err, pion=pion, muon=muon)


def plot_pion_muon_offset(langau, hfit, tpc=-1, params=OFFSET_PARAMS, window=None,
                          max_rel_err=None, x_limits=None):
    """For each parameter, per plane: pion slices and offset muon slices vs pion residual
    range (top), and pion - (muon + offset) over the overlap (bottom).
    Returns the offsets, their errors and chi2/ndf as a DataFrame."""
    colors = {"MC": "#d62728", "Data": "#1f77b4"}
    tpc_name = "both TPCs" if tpc == -1 else f"TPC {tpc}"
    rows = []
    for p_col, p_err, ylabel in params:
        fig, axes = plt.subplots(2, 3, figsize=(18, 10), height_ratios=(3, 1),
                                 layout="constrained")
        for plane in range(3):
            ax, ax_d = axes[0, plane], axes[1, plane]
            text = []
            for sample, color in colors.items():
                pi_res = langau.get(("pion", sample), {}).get((plane, tpc))
                mu_res = langau.get(("muon", sample), {}).get((plane, tpc))
                if (pi_res is None or mu_res is None or not len(pi_res) or not len(mu_res)
                        or p_col not in pi_res or p_col not in mu_res):
                    continue
                o = pion_muon_offset(hfit, pi_res, mu_res, p_col, p_err, window, max_rel_err)
                if o is None:
                    print(f"{p_col}, {sample}, plane {plane}: no pion/muon overlap")
                    continue
                (x_pi, y_pi, e_pi), (x_mu, y_mu, e_mu) = o["pion"], o["muon"]
                ax.errorbar(x_pi, y_pi, yerr=e_pi, fmt="o", ms=3.5, color=color,
                            elinewidth=0.7, alpha=0.85, label=f"Pion {sample}")
                ax.errorbar(x_mu, y_mu + o["offset"], yerr=e_mu, fmt="s", ms=3.5, color=color,
                            mfc="none", elinewidth=0.7, alpha=0.85,
                            label=f"Muon {sample} + offset (equal $\\beta\\gamma$)")
                ax_d.errorbar(o["x"], o["diff"] - o["offset"], yerr=o["diff_err"], fmt="o",
                              ms=3, color=color, elinewidth=0.7, alpha=0.85)
                text.append(rf"{sample}: offset {o['offset']:+.4f} $\pm$ {o['offset_err']:.4f}, "
                            rf"$\chi^2$/NDF = {o['chi2']:.1f}/{o['ndf']}")
                rows.append(dict(param=p_col, sample=sample, plane=plane, tpc=tpc,
                                 offset=o["offset"], offset_err=o["offset_err"],
                                 chi2=o["chi2"], ndf=o["ndf"], chi2_ndf=o["chi2"] / o["ndf"],
                                 n_window=o["n_window"]))

            if window is not None:
                for a in (ax, ax_d):
                    a.axvspan(*window, color="gray", alpha=0.12, label="Offset window")
            ax_d.axhline(0.0, color="gray", lw=1)
            if text:
                ax.text(0.97, 0.97, "\n".join(text), transform=ax.transAxes, fontsize=10,
                        va="top", ha="right",
                        bbox=dict(boxstyle="round,pad=0.3", facecolor="white", alpha=0.85,
                                  edgecolor="none"))
            if x_limits is not None:
                ax.set_xlim(x_limits)
            ax.set_title(f"Plane {plane}", fontsize=18)
            ax_d.set_xlabel("Pion residual range [cm]", fontsize=16)
            for a in (ax, ax_d):
                a.tick_params(labelsize=13)
                a.grid(True, ls=":", alpha=0.5)
        axes[0, 0].set_ylabel(ylabel, fontsize=16)
        axes[1, 0].set_ylabel("Pion $-$ (muon + offset)", fontsize=14)
        handles, labels = axes[0, 0].get_legend_handles_labels()
        fig.legend(handles, labels, loc="outside lower center", ncol=5, fontsize=12)
        fig.suptitle(f"{ylabel.split(' [')[0]}, {tpc_name}", fontsize=18)
        plt.show()
    return pd.DataFrame(rows)


offsets = plot_pion_muon_offset(langau, hfit)                    # offset from the whole overlap
# offsets = plot_pion_muon_offset(langau, hfit, window=(20, 60)) # offset from a window only
